In [ ]:
!date

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import scanpy as sc
import anndata

from pybedtools import BedTool

In [ ]:
sc.set_figure_params(figsize=(5,5), frameon=False)
sc.settings.verbosity = 3

In [ ]:
projdir = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/mc'
donors = list(np.loadtxt(f'{projdir}/txt/donors.txt', dtype=str))
tmp_order = ['Start', 'Sendai', 'Fail1', 'Delayed', 'Inter1', 'Inter2', 'Fail2', 'IPS']
clusts = [f'{x}_{y}' for x,y in zip(np.repeat(tmp_order, len(donors)), np.tile(donors, len(tmp_order)))]

In [ ]:
genes_path = '/u/project/cluo/terencew/reference/hg38_igvf/bed/genes/gencode.v43.autosome.dedup.names.bed.gz'
genes = pd.read_csv(genes_path, sep='\t', header=None)
genes.index = genes[3]
genes.shape

In [ ]:
genes.head()

In [ ]:
degs_bin_path = '/u/project/cluo/Shared_Datasets/IGVF/202208_Pilot/new_multiome/YS3v7_Data/YS3v7.Path_IPS.DEGs_IPS_with_PWM.Bins.table.txt'
degs_bin = pd.read_csv(degs_bin_path, sep='\t')
degs_bin.shape

In [ ]:
degs_bin.head()

In [ ]:
degs_path = '/u/project/cluo/Shared_Datasets/IGVF/202208_Pilot/new_multiome/YS3v7_Data/YS3v7.Branch_IPS.DEGs.table.txt'
degs = pd.read_csv(degs_path, sep='\t')
degs.shape

In [ ]:
degs.head()

In [ ]:
indir = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/pseudobulk_allc/pmds/dnmtools/rna_cluster_donor'
pmds = pd.read_csv(f'{indir}/merged/start_merged.filt_mcg.sorted.bed.gz', sep='\t', header=None)
pmds.index = [f'{x}_{y}' for x,y in zip(pmds[0], pmds[1])]
pmds.columns = ['chrom', 'start', 'end']
pmds.shape

### bin by bin

In [ ]:
tmp_bins = np.arange(10) + 1

gene_stats = []
for s in tmp_bins:
    tmp_genes = degs_bin[degs_bin['PWM.bin'] == s].index
    tmp_genes = tmp_genes[tmp_genes.isin(genes.index)]
    bin_genes = genes.loc[tmp_genes]
    pmds_bt = BedTool.from_dataframe(pmds)
    genes_bt = BedTool.from_dataframe(bin_genes)
    genes_inter = genes_bt.intersect(pmds_bt).to_dataframe()
    n_genes = genes_inter.shape[0]
    total_genes = tmp_genes.shape[0]
    gene_stats.append([n_genes, total_genes])
    
gene_df = pd.DataFrame(gene_stats)
gene_df.columns = ['num_genes_pmds', 'num_genes']
gene_df['pmd_frac'] = gene_df['num_genes_pmds'] / gene_df['num_genes']
gene_df['bin'] = gene_df.index
gene_df

In [ ]:
ax = sns.barplot(data=gene_df, x='bin', y='pmd_frac', hue='bin')

In [ ]:
outdir = '/u/project/cluo/terencew/igvf/2023_YR2/final_figures/csv/figure4'
gene_df.to_csv(f'{outdir}/ips_binned_genes_pmds.csv', sep='\t')

In [ ]:
!date